# Encodage du probleme
- SEND + MORE = MONEY
- Variables : S E N D M O R Y  (8 chiffres distincts, S != 0, M != 0)

In [2]:
# Imports
import time
from itertools import permutations

In [3]:
VARIABLES = ['S', 'E', 'N', 'D', 'M', 'O', 'R', 'Y']
DOMAIN    = list(range(10))

def check_constraint(assignment):
    """Verifie la contrainte SEND + MORE = MONEY pour une assignation complete ou partielle."""
    val = assignment.get

    S, E, N, D = val('S'), val('E'), val('N'), val('D')
    M, O, R, Y = val('M'), val('O'), val('R'), val('Y')

    # Contraintes unaires : premiers chiffres non nuls
    if S is not None and S == 0:
        return False
    if M is not None and M == 0:
        return False

    # Contrainte globale uniquement si toutes les variables sont assignees
    assigned = [v for v in VARIABLES if val(v) is not None]
    if len(assigned) < len(VARIABLES):
        return True  # pas encore completement assigne, pas de violation detectee

    SEND  = S * 1000 + E * 100 + N * 10 + D
    MORE  = M * 1000 + O * 100 + R * 10 + E
    MONEY = M * 10000 + O * 1000 + N * 100 + E * 10 + Y

    return SEND + MORE == MONEY

# Methode 1 : Generate and Test (GT)
# On genere toutes les permutations possibles et on teste chaque candidate.

def generate_and_test():
    solutions = []

    # On parcourt toutes les permutations de 8 chiffres parmi 0..9
    for perm in permutations(DOMAIN, len(VARIABLES)):
        assignment = dict(zip(VARIABLES, perm))

        if assignment['S'] == 0 or assignment['M'] == 0:
            continue

        S, E, N, D = assignment['S'], assignment['E'], assignment['N'], assignment['D']
        M, O, R, Y = assignment['M'], assignment['O'], assignment['R'], assignment['Y']

        SEND  = S * 1000 + E * 100 + N * 10 + D
        MORE  = M * 1000 + O * 100 + R * 10 + E
        MONEY = M * 10000 + O * 1000 + N * 100 + E * 10 + Y

        if SEND + MORE == MONEY:
            solutions.append(assignment.copy())

    return solutions

# Methode 2 : Backtracking (BT)
# On assigne les variables une par une avec elagage precoce.

def is_consistent(var, value, assignment):
    """Verifie la coherence locale apres avoir assigne var = value."""
    # Unicite des valeurs
    if value in assignment.values():
        return False

    # Contraintes unaires
    if var in ('S', 'M') and value == 0:
        return False

    # Contrainte partielle sur les colonnes (propagation anticipee)
    test = assignment.copy()
    test[var] = value

    # Si on a assigne toutes les variables, on verifie la contrainte globale
    if len(test) == len(VARIABLES):
        S, E, N, D = test['S'], test['E'], test['N'], test['D']
        M, O, R, Y = test['M'], test['O'], test['R'], test['Y']
        SEND  = S * 1000 + E * 100 + N * 10 + D
        MORE  = M * 1000 + O * 100 + R * 10 + E
        MONEY = M * 10000 + O * 1000 + N * 100 + E * 10 + Y
        return SEND + MORE == MONEY

    return True


def backtrack(assignment, solutions):
    if len(assignment) == len(VARIABLES):
        solutions.append(assignment.copy())
        return

    # Selection de la prochaine variable non assignee (ordre fixe)
    var = next(v for v in VARIABLES if v not in assignment)

    for value in DOMAIN:
        if is_consistent(var, value, assignment):
            assignment[var] = value
            backtrack(assignment, solutions)
            del assignment[var]


def backtracking():
    solutions = []
    backtrack({}, solutions)
    return solutions

In [4]:

# Execution et comparaison

def format_solution(sol):
    S, E, N, D = sol['S'], sol['E'], sol['N'], sol['D']
    M, O, R, Y = sol['M'], sol['O'], sol['R'], sol['Y']
    SEND  = S * 1000 + E * 100 + N * 10 + D
    MORE  = M * 1000 + O * 100 + R * 10 + E
    MONEY = M * 10000 + O * 1000 + N * 100 + E * 10 + Y
    mapping = ' '.join(f"{k}={v}" for k, v in sol.items())
    return f"{mapping}  =>  {SEND} + {MORE} = {MONEY}"


def run_method(name, func):
    print(f"\n{'='*55}")
    print(f" Methode : {name}")
    print(f"{'='*55}")
    start = time.perf_counter()
    solutions = func()
    elapsed = time.perf_counter() - start
    print(f" Temps d'execution : {elapsed:.4f} secondes")
    print(f" Nombre de solutions trouvees : {len(solutions)}")
    for sol in solutions:
        print(f"   {format_solution(sol)}")
    return elapsed, solutions


if __name__ == "__main__":
    print("Probleme CSP : SEND + MORE = MONEY")
    print("Variables :", VARIABLES)
    print("Domaine   : chiffres 0 a 9, tous distincts")
    print("Contrainte: S != 0, M != 0")

    t_gt, sol_gt = run_method("Generate and Test (GT)", generate_and_test)
    t_bt, sol_bt = run_method("Backtracking       (BT)", backtracking)

    print(f"\n{'='*55}")
    print(" Comparaison des performances")
    print(f"{'='*55}")
    print(f" GT  : {t_gt:.4f} s")
    print(f" BT  : {t_bt:.4f} s")

    if t_bt < t_gt:
        ratio = t_gt / t_bt if t_bt > 0 else float('inf')
        print(f" BT est {ratio:.1f}x plus rapide que GT")
    else:
        ratio = t_bt / t_gt if t_gt > 0 else float('inf')
        print(f" GT est {ratio:.1f}x plus rapide que BT")

    print(f"\n Les deux methodes trouvent les memes solutions : {sol_gt == sol_bt}")

Probleme CSP : SEND + MORE = MONEY
Variables : ['S', 'E', 'N', 'D', 'M', 'O', 'R', 'Y']
Domaine   : chiffres 0 a 9, tous distincts
Contrainte: S != 0, M != 0

 Methode : Generate and Test (GT)
 Temps d'execution : 1.9607 secondes
 Nombre de solutions trouvees : 1
   S=9 E=5 N=6 D=7 M=1 O=0 R=8 Y=2  =>  9567 + 1085 = 10652

 Methode : Backtracking       (BT)
 Temps d'execution : 3.2748 secondes
 Nombre de solutions trouvees : 1
   S=9 E=5 N=6 D=7 M=1 O=0 R=8 Y=2  =>  9567 + 1085 = 10652

 Comparaison des performances
 GT  : 1.9607 s
 BT  : 3.2748 s
 GT est 1.7x plus rapide que BT

 Les deux methodes trouvent les memes solutions : True
